# ie-agent walkthrough: one document, step by step

This notebook shows what happens inside the agent for **one document**. Part 2 plays the model's role by hand, so
every step (which tools are offered, what a tool returns, how the stage changes) is visible without a GPU. Part 3
lets the real model do the same, **one request per cell**: what the model gets, what it answers, what the
loop does with the answer.

**Needs:** Weaviate (port 8087) and TEI (port 8090) for `map_to_gnd`, the GND data (`uv run ie-agent build-gnd`).
Part 3 also needs a vLLM server.

Kernel: `ie-agent/.venv` (in VS Code: *Select Kernel* → *Python Environments* → `ie-agent/.venv`).

In [1]:
import json
import textwrap
from types import SimpleNamespace

import yaml

from ie_agent.agent import Agent
from ie_agent.config import CONFIG
from ie_agent.data import Documents, Mapper, load_gold
from ie_agent.gnd import Gnd
from ie_agent.tools import allowed, check_call, new_state, offered_tools, stage

DOC_ID = "1001811895"   # change to any of the 827 validate documents


def show(text, width=110, limit=1500):
    """Prints long text wrapped and shortened."""
    text = str(text)
    print(textwrap.fill(text[:limit], width) + (" …" if len(text) > limit else ""))


def names(tools):
    return [t["function"]["name"] for t in tools]


def show_state(state):
    """The parts of the state that decide what comes next."""
    print(f"   state: stage {stage(state)} | read {sorted(state['read'])} | mapped {len(state['mapped'])} | "
          f"shown by gnd_context {len(state['graph'])} | gnd_context calls {state['context_calls']}")


def show_message(m, limit=4000):
    """One chat message as the model receives it (tool results that are JSON are indented)."""
    print(f"----- {m['role'].upper()} -----")
    content = m.get("content") or ""
    try:
        content = json.dumps(json.loads(content), ensure_ascii=False, indent=2)
    except (json.JSONDecodeError, TypeError):
        pass
    print(content[:limit] + (" …" if len(content) > limit else ""))

## Part 1: the data the agent works with

`data.py` loads everything once: titles, sections (abstract/introduction, table of contents), the few-shot
examples of the pipeline runs, the GND graph with definitions, and the mapper (Weaviate + TEI).

In [2]:
cfg = yaml.safe_load(CONFIG.read_text())
docs = Documents(cfg)
gnd = Gnd.load(cfg)
mapper = Mapper(cfg)
gold = load_gold(cfg)

print("documents:", len(docs.ids), "| GND terms:", len(gnd.g["pref"]), "| with details:", len(gnd.details))
print()
print("TITLE:", docs.title[DOC_ID])
print("SOURCE of the abstract section:", docs.source[DOC_ID])
print("GOLD (what the indexers assigned):", sorted(gnd.label(i) for i in gold[DOC_ID]))

documents: 827 | GND terms: 309417 | with details: 197252

TITLE: Semantisches Informationsmodell für die Betriebsunterstützung dienstorientierter Systeme
SOURCE of the abstract section: Einleitung
GOLD (what the indexers assigned): ['Dienstgüte', 'Informationsmanagement', 'Informationsmodell', 'Semantic Web', 'Servicegrad', 'Serviceorientierte Architektur', 'Softwarearchitektur', 'Web Services']


## Part 2: one run by hand

### 2.1 What the model gets at the start

With every request the model gets three things, and nothing else of the document:

1. the **system prompt** (same for all documents),
2. the **messages** so far; at the start only the **first message** with the title and five similar, already
   indexed titles,
3. the **tool definitions** (JSON schemas) of the tools offered in this step.

#### 1. The system prompt

`Agent.__init__` (`agent.py`) puts it together: `system.md` with the placeholders filled from
`system_parts.yaml` (they depend on the conditions), then `gnd_context.md` + `gnd_context_ie.md` if
`gnd_context` is on (and `examples_gnd.md` for variant A).

In [3]:
import re
from ie_agent.agent import PROMPTS

# an agent without a server: we only use its prompts and its tools here
cfg_offline = {**cfg, "llm": {**cfg["llm"], "model": "(none)"}}
agent = Agent(cfg_offline, docs, mapper, gnd, client=SimpleNamespace())

print("placeholders in system.md:", re.findall(r"\{(\w+)\}", (PROMPTS / "system.md").read_text()))
print("conditions:", {k: cfg["agent"][k] for k in ["gnd_context", "toc", "examples_gnd", "examples_with_sections"]})
print()
print(agent.system_prompt)

placeholders in system.md: ['toc_step', 'examples_step', 'check_step', 'finish_number', 'id_sources']
conditions: {'gnd_context': True, 'toc': True, 'examples_gnd': False, 'examples_with_sections': False}

Du bist Fachreferent:in für die verbale Sacherschließung an der Deutschen Nationalbibliothek und vergibst Schlagwörter aus der Gemeinsamen Normdatei (GND) für wissenschaftliche Arbeiten.

Du bekommst den Titel eines Dokuments und Beispiele ähnlicher, bereits erschlossener Titel. Gehe stufenweise vor, wie es Erschließer:innen tun:

1. Bestimme aus dem Titel das Hauptthema.
2. Reicht der Titel nicht aus (zu allgemein, metaphorisch, mehrdeutig, Fachbegriffe unklar), lies mit read_abstract_intro() Gegenstand und Fragestellung.
3. Bilde deine Kandidaten aus Titel und Abstract mit map_to_gnd() ab. Fehlt danach noch etwas, lies mit read_toc() Aufbau und behandelte Aspekte. Das Inhaltsverzeichnis steht erst zur Verfügung, wenn du nach dem Abstract abgebildet hast. Lies nur so weit, wie du es

#### 2. The first message

`prompts/first_message.md` with the title and the five few-shot examples of the title condition.

In [4]:
first_message = agent.first_message.format(title=docs.title[DOC_ID], examples=agent.tools.examples(DOC_ID, "title"))
print(first_message)

Titel: Semantisches Informationsmodell für die Betriebsunterstützung dienstorientierter Systeme

Ähnliche erschlossene Titel:

Beispiel 1: Semantisches Informationsmodell für die Betriebsunterstützung dienstorientierter Systeme
Schlagwörter: Informationsmanagement; Softwarearchitektur; Servicegrad; Informationsmodell; Dienstgüte; Semantic Web; Web Services; Serviceorientierte Architektur

Beispiel 2: Dienstgüteunterstützung für Service-orientierte Workflows
Schlagwörter: Prozessmanagement; Dienstgüte; Web Services; Serviceorientierte Architektur

Beispiel 3: Ein modellgetriebener Ansatz zur Entwicklung informationsflusssicherer Systeme
Schlagwörter: Informationsfluss; Mobiles Endgerät; App (Programm); Informationssicherheit; Web Services; Modellgetriebene Entwicklung

Beispiel 4: Informationsmodell zur funktionalen Typisierung von Automatisierungsgeräten. Information model for a functional typificaion of automation devices
Schlagwörter: Automatisierungssystem; Ethernet; Feldbus

#### 3. The tool definitions

All the model knows about a tool is its schema: name, description, parameters. `offered_tools` (`tools.py`)
picks the schemas for the current state, so the model does not even see the tools it may not use yet. These
are the ones of the first request:

In [5]:
print(json.dumps(offered_tools(new_state(docs.title[DOC_ID]), last_step=False), ensure_ascii=False, indent=2))

[
  {
    "type": "function",
    "function": {
      "name": "read_abstract_intro",
      "description": "Liefert das Abstract des Dokuments oder, falls es keines gibt, den Anfang der Einleitung.",
      "parameters": {
        "type": "object",
        "properties": {},
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "map_to_gnd",
      "description": "Bildet freie Schlagwort-Kandidaten auf GND-Schlagwörter ab. Liefert pro Kandidat GND-ID, Ansetzungsform und einen Score (1,0 = exakter Treffer).",
      "parameters": {
        "type": "object",
        "properties": {
          "candidates": {
            "type": "array",
            "items": {
              "type": "string"
            }
          }
        },
        "required": [
          "candidates"
        ]
      }
    }
  }
]


### 2.2 The state and the IE policy

For each document the agent keeps a **state**: what was read, what was mapped, which IDs the GND graph showed.
The **stage** is derived from it, and the stage decides which tools the model is offered:

```
TITLE ──read_abstract_intro──▶ ABSTRACT ──map_to_gnd──▶ MAPPED ──read_toc──▶ TOC
```

In [6]:
state = new_state(docs.title[DOC_ID])
print("stage:", stage(state))
print("offered tools:", names(offered_tools(state, last_step=False)))
print("in the forced last step:", names(offered_tools(state, last_step=True)))

stage: title
offered tools: ['read_abstract_intro', 'map_to_gnd']
in the forced last step: ['read_abstract_intro', 'map_to_gnd', 'finish']


At the start only `read_abstract_intro` and `map_to_gnd` are offered: the table of contents is locked,
`gnd_context` needs mapped candidates first, `finish` needs a `gnd_context` call first.

Every tool call goes through two functions:
- `check_call` (policy): is the call allowed now? If not, the model gets a message instead of a result.
- `agent.tools.run`: the tool itself.

### 2.3 Step 1: the title is not enough → `read_abstract_intro`

In [7]:
def call(name, **args):
    """What happens when the model calls a tool: policy check first, then the tool, then the state."""
    refused = check_call(name, args, state)
    result = refused or agent.tools.run(DOC_ID, name, args, state)
    print(f"--> {name}({json.dumps(args, ensure_ascii=False)})", "REFUSED" if refused else "")
    show_state(state)
    print()
    return result


result = call("read_abstract_intro")
show(result)
print()
print("offered now:", names(offered_tools(state, last_step=False)))

--> read_abstract_intro({}) 
   state: stage abstract | read ['abstract_intro'] | mapped 0 | shown by gnd_context 0 | gnd_context calls 0

[Einleitung] Bei der Umsetzung von Geschäftsmodellen kommt der Geschwindigkeit, mit der ein Unternehmen
Veränderungen  am  Markt  erkennt  und  ein  innovatives  Geschäftskonzept realisiert, eine große Bedeutung zu
(Eisenhardt und Brown 1998; Economist Intelligence 2006). Eng damit verbunden ist die Fähigkeit eines
Unternehmens seine Geschäftsprozesse  und  die  hierdurch  erbrachten  Dienstleistungen  flexibel  an  die
aktuellen  Rahmenbedingungen  und  Bedürfnisse  anzupassen  (Genovese,  Hayward, Phifer,  Plummer  et  al.
2005).  Vor  diesem  Hintergrund  kommt  der  Informationstechnologie  (IT)  bei  der  softwaretechnischen
Realisierung  effizienter  und  flexibler Geschäftsprozesse eine zentrale Rolle zu (Kossmann und Leymann
2004).  In der  Realität  sind  die  IT-Landschaften  von  Unternehmen  allerdings  oft  über Jahrzehnte
gewachsene un

The policy also protects the order: asking for the table of contents now is refused, because the agent has
not mapped anything since reading the abstract.

In [8]:
print(call("read_toc"))

--> read_toc({}) REFUSED
   state: stage abstract | read ['abstract_intro'] | mapped 0 | shown by gnd_context 0 | gnd_context calls 0

read_toc steht jetzt nicht zur Verfügung. Reihenfolge: Titel, dann read_abstract_intro(), dann map_to_gnd(), dann bei Bedarf read_toc(); jeder Abschnitt wird nur einmal gelesen.


### 2.4 Step 2: map candidates onto the GND → `map_to_gnd`

The model writes free candidates. Each one is embedded (TEI) and searched in the GND vocabulary (Weaviate, hybrid
search). Hits with a cosine similarity below 0.7 are dropped. Only mapped IDs may be assigned later.

Look at the score the model sees (Weaviate's hybrid score, not the cosine similarity): 1.0 is an exact hit. "Betriebsunterstützung" and "IT-Management" only reach 0.7. They were
matched via a variant name of another term ("Finanzielle Unterstützung", "IT-System"), which is a warning sign.

In [9]:
candidates = ["Informationsmodell", "Serviceorientierte Architektur", "Semantic Web", "Betriebsunterstützung",
              "Dienstgüte", "IT-Management"]
result = call("map_to_gnd", candidates=candidates)
for hit in json.loads(result):
    print(hit)
print()
print("offered now:", names(offered_tools(state, last_step=False)))

--> map_to_gnd({"candidates": ["Informationsmodell", "Serviceorientierte Architektur", "Semantic Web", "Betriebsunterstützung", "Dienstgüte", "IT-Management"]}) 
   state: stage mapped | read ['abstract_intro'] | mapped 6 | shown by gnd_context 0 | gnd_context calls 0

{'kandidat': 'Informationsmodell', 'gnd_id': '94875463X', 'gnd_begriff': 'Informationsmodell', 'score': 1.0}
{'kandidat': 'Serviceorientierte Architektur', 'gnd_id': '976266202', 'gnd_begriff': 'Serviceorientierte Architektur', 'score': 1.0}
{'kandidat': 'Semantic Web', 'gnd_id': '964629119', 'gnd_begriff': 'Semantic Web', 'score': 1.0}
{'kandidat': 'Betriebsunterstützung', 'gnd_id': '990782700', 'gnd_begriff': 'Finanzielle Unterstützung', 'score': 0.7}
{'kandidat': 'Dienstgüte', 'gnd_id': '953301591', 'gnd_begriff': 'Dienstgüte', 'score': 1.0}
{'kandidat': 'IT-Management', 'gnd_id': '041252292', 'gnd_begriff': 'IT-System', 'score': 0.7}

offered now: ['read_toc', 'get_examples', 'map_to_gnd', 'gnd_context']


**Two numbers per hit.** `Mapper.map` (`data.py`) returns both. `ToolRunner.map` (`tools.py`) decides with the
**cosine similarity** of the two embeddings (`cosine_similarity >= min_cosine`, 0.7), but the model is shown the
**hybrid score** of Weaviate (word search + vector search, `alpha` 0.7). The rule "score below 1.0 = check it with
gnd_context" in the prompt means the hybrid score.

In [10]:
for c in candidates:
    m = mapper.map(c)
    verdict = "accepted" if m["cosine_similarity"] >= cfg["mapping"]["min_cosine"] else "dropped"
    print(f"{c:32s} -> {m['term']:32s} hybrid {m['score']:.3f} | cosine {m['cosine_similarity']:.3f} | {verdict}")

Informationsmodell               -> Informationsmodell               hybrid 1.000 | cosine 1.000 | accepted
Serviceorientierte Architektur   -> Serviceorientierte Architektur   hybrid 1.000 | cosine 1.000 | accepted
Semantic Web                     -> Semantic Web                     hybrid 1.000 | cosine 1.000 | accepted
Betriebsunterstützung            -> Finanzielle Unterstützung        hybrid 0.700 | cosine 0.779 | accepted
Dienstgüte                       -> Dienstgüte                       hybrid 1.000 | cosine 1.000 | accepted
IT-Management                    -> IT-System                        hybrid 0.700 | cosine 0.833 | accepted


### 2.5 Step 3: something is missing → `read_toc`

Mapping after the abstract unlocked the table of contents (stage MAPPED). It shows which aspects the work
covers.

In [11]:
show(call("read_toc"), limit=1200)

--> read_toc({}) 
   state: stage toc | read ['abstract_intro', 'toc'] | mapped 6 | shown by gnd_context 0 | gnd_context calls 0

ZENTRALE FRAGESTELLUNGEN UND BEITRÄGE DIENSTORIENTIERTE ARCHITEKTUREN UND SYSTEME Szenario: Der KIT-Campusplan
Betriebliche Herausforderungen dienstorientierter Systeme Dimension: Heterogenität des verteilten
Informationsraums Dimension: Isolierte Betrachtungsweise Dimension: Definition und Durchsetzung von
Dienstgütevereinbarungen ANSÄTZE IM BEREICH 'HOMOGENISIERTER INFORMATIONSRAUM' Common Data Model (CDM) Common
Information Model (CIM) Vereinheitlichte Spezifikation von Fachkomponenten nach (Ackermann et al. 2002)
Bewertung ANSÄTZE IM BEREICH 'HOLISTISCHE DIENSTSICHTEN' 3-Layer Graph-Based Meta-Model (3LGM²) Software
Cartography Tool (SoCaTool) BMC Remedy CI Relationship Viewer ANSÄTZE IM BEREICH 'KONSISTENTE
DIENSTGÜTEVEREINBARUNGEN' Web Service Level Agreement (WSLA) Semantisches Dienstgütemodell nach (Mabrouk,
Georgantas und Issarny 2009) BMC Service L

The table of contents mentions web services and software architecture, so the model maps two more
candidates.

In [12]:
for hit in json.loads(call("map_to_gnd", candidates=["Web Services", "Softwarearchitektur"])):
    print(hit)
print()
print("mapped so far:", {i: m["term"] for i, m in state["mapped"].items()})

--> map_to_gnd({"candidates": ["Web Services", "Softwarearchitektur"]}) 
   state: stage toc | read ['abstract_intro', 'toc'] | mapped 8 | shown by gnd_context 0 | gnd_context calls 0

{'kandidat': 'Web Services', 'gnd_id': '964788543', 'gnd_begriff': 'Web Services', 'score': 1.0}
{'kandidat': 'Softwarearchitektur', 'gnd_id': '041216776', 'gnd_begriff': 'Softwarearchitektur', 'score': 1.0}

mapped so far: {'94875463X': 'Informationsmodell', '976266202': 'Serviceorientierte Architektur', '964629119': 'Semantic Web', '990782700': 'Finanzielle Unterstützung', '953301591': 'Dienstgüte', '041252292': 'IT-System', '964788543': 'Web Services', '041216776': 'Softwarearchitektur'}


### 2.6 Step 4: check the candidates in the GND → `gnd_context` (gnd_context)

For each ID the agent gets what the GND knows: definition or description, using instructions, subject category,
broader, narrower and related terms, how often the term was assigned in training, and how it relates to the
other candidates. Narrower terms that occur in the text read so far are marked `im_text`.

In [13]:
ids = list(state["mapped"])
result = json.loads(call("gnd_context", gnd_ids=ids))
print(json.dumps(result[0], ensure_ascii=False, indent=2))

--> gnd_context({"gnd_ids": ["94875463X", "976266202", "964629119", "990782700", "953301591", "041252292", "964788543", "041216776"]}) 
   state: stage toc | read ['abstract_intro', 'toc'] | mapped 8 | shown by gnd_context 25 | gnd_context calls 1

{
  "gnd_id": "94875463X",
  "begriff": "Informationsmodell",
  "definition": null,
  "sachgruppen": [
    "Informatik, Datenverarbeitung"
  ],
  "in_trainingsdokumenten_vergeben": 11,
  "oberbegriffe": [],
  "unterbegriffe": {
    "anzahl": 0,
    "auswahl": []
  },
  "verwandte_begriffe": [],
  "beziehungen_zu_deinen_kandidaten": []
}


In [14]:
for entry in result:
    if "begriff" not in entry:
        continue
    narrower = [n["begriff"] + (" (im Text)" if n.get("im_text") else "") for n in entry["unterbegriffe"]["auswahl"]]
    print(f"{entry['begriff']:32s} assigned {entry['in_trainingsdokumenten_vergeben']:>5}x | "
          f"categories {entry['sachgruppen']} | narrower {narrower[:4]}")
    if entry["beziehungen_zu_deinen_kandidaten"]:
        print(" " * 34, entry["beziehungen_zu_deinen_kandidaten"])
print()
print("IDs the graph showed (may now be assigned too):", len(state["graph"]))
print("finish offered now:", "finish" in names(offered_tools(state, last_step=False)))

Informationsmodell               assigned    11x | categories ['Informatik, Datenverarbeitung'] | narrower []
Serviceorientierte Architektur   assigned    14x | categories ['Informatik, Datenverarbeitung'] | narrower []
Semantic Web                     assigned     4x | categories ['Informatik, Datenverarbeitung', 'Bibliothek, Information und Dokumentation'] | narrower ['SPARQL', 'Linked Data', 'World Wide Web 3.0']
Finanzielle Hilfe                assigned     1x | categories ['Sozialarbeit, Sozialhilfe'] | narrower []
Dienstgüte                       assigned    12x | categories ['Informatik, Datenverarbeitung', 'Elektronik, Kommunikationstechnik'] | narrower []
Datenverarbeitungssystem         assigned     1x | categories ['Informatik, Datenverarbeitung'] | narrower ['Anwendungssystem (im Text)', 'Verteiltes System', 'Echtzeitsystem', 'Objektverwaltung']
Web Services                     assigned     5x | categories ['Informatik, Datenverarbeitung'] | narrower ['Geodienst']
Softwarea

Here `gnd_context` does its job: the two fuzzy hits turn out to be "Finanzielle Hilfe" (category
*Sozialarbeit, Sozialhilfe*) and "Datenverarbeitungssystem" (assigned once in training). Neither fits a work on
service-oriented systems, so the model should drop them, as the gnd_context prompt says: *Passt der GND-Begriff laut
Definition und Sachgruppe wirklich zum Dokument? Sonst weglassen.*

### 2.7 Step 5: `finish`

`finish` is not a normal tool: the loop in `agent.py` checks the IDs. Only IDs that came from `map_to_gnd` or
`gnd_context` are allowed. An ID from the model's memory is sent back (at most twice) with this message:

In [15]:
from ie_agent.agent import rejection

# keep the exact hits, drop the two fuzzy ones that gnd_context showed to be off-topic
chosen = [i for i, m in state["mapped"].items() if m["score"] == 1.0]
made_up = "4044038-4"   # a GND number the model "remembers"
print("allowed:", {i: allowed(state, i) for i in chosen + [made_up]})
print()
print(rejection([made_up], chosen))

allowed: {'94875463X': True, '976266202': True, '964629119': True, '953301591': True, '964788543': True, '041216776': True, '4044038-4': False}

Nicht abgeschlossen. Diese IDs stammen nicht aus map_to_gnd: 4044038-4. Verwende nur GND-IDs aus map_to_gnd, auch wenn du eine GND-Nummer zu kennen glaubst. Bilde deine Kandidaten zuerst mit map_to_gnd ab und rufe dann finish erneut auf. Gültig waren: 94875463X, 976266202, 964629119, 953301591, 964788543, 041216776.


### 2.8 How good was this hand-made result?

Same measure as `ie-agent eval`: set precision, recall and F1 against the gold of the indexers.

In [16]:
def score(predicted, doc_id=DOC_ID):
    predicted, g = set(predicted), gold[doc_id]
    hits = len(predicted & g)
    p = hits / len(predicted) if predicted else 0.0
    r = hits / len(g)
    f1 = 2 * p * r / (p + r) if hits else 0.0
    print(f"P {p:.2f} | R {r:.2f} | F1 {f1:.2f}")
    print("hits:  ", sorted(gnd.label(i) for i in predicted & g))
    print("extra: ", sorted(gnd.label(i) for i in predicted - g))
    print("missed:", sorted(gnd.label(i) for i in g - predicted))


score(chosen)
print()
print("with the two fuzzy hits it would have been:")
score(state["mapped"])

P 1.00 | R 0.75 | F1 0.86
hits:   ['Dienstgüte', 'Informationsmodell', 'Semantic Web', 'Serviceorientierte Architektur', 'Softwarearchitektur', 'Web Services']
extra:  []
missed: ['Informationsmanagement', 'Servicegrad']

with the two fuzzy hits it would have been:
P 0.75 | R 0.75 | F1 0.75
hits:   ['Dienstgüte', 'Informationsmodell', 'Semantic Web', 'Serviceorientierte Architektur', 'Softwarearchitektur', 'Web Services']
extra:  ['Datenverarbeitungssystem', 'Finanzielle Hilfe']
missed: ['Informationsmanagement', 'Servicegrad']


## Part 3: the real model, one request at a time

Now the model decides itself. `agent.annotate()` would run the whole loop at once. The cells below run the loop
of `Agent._loop` (`agent.py`) **one request per `next_step()`**, with the agent's own helpers (`offered_tools`,
`_run_tool`, `_finish`, `_result`). Each request prints three blocks:

1. **MODEL GETS**: the messages that are new since the last request (at the first request: system prompt and
   first message), the offered tools and whether `finish` is forced. The model always gets the *whole*
   conversation; only the new part is printed.
2. **MODEL ANSWERS**: its reasoning (if on), text and tool calls.
3. **WHAT HAPPENS**: each tool call through `check_call` and the tool; the tool result is what the model gets at
   the next request. Then the state.

Needs a running vLLM server (Qwen: `--tool-call-parser qwen3_coder --reasoning-parser qwen3`).

The live run uses another document than Part 2: document 1001811895 has a duplicate in the training set (same
thesis, other IDN), so its first few-shot example is the document itself with all gold headings, and the model
would just copy them.

In [17]:
PORT = 9517                  # vLLM server: models: in config.yaml, 9516 + N when started with --gpu N
THINKING = True              # reasoning on (then max_tokens is raised to 8192)
LIVE_DOC_ID = "1007748524"   # "Optische Spektroskopie an zweidimensionalen Lochsystemen ...", no duplicate
REASONING_CHARS = 3000       # how much of the reasoning to print per request

live_cfg = {**cfg, "llm": {**cfg["llm"], "base_url": f"http://localhost:{PORT}/v1", "reasoning": THINKING,
                           "model": None, "max_tokens": 8192 if THINKING else 2048}}
live_agent = Agent(live_cfg, docs, mapper, gnd)   # model None: asks the server which model it serves
print("model:", live_agent.model, "| prompt hash:", live_agent.prompt_hash)
print("document:", docs.title[LIVE_DOC_ID])

model: Qwen/Qwen3.8-27B-FP8 | prompt hash: 3950928c62c4
document: Optische Spektroskopie an zweidimensionalen Lochsystemen in GaAs-AlGaAs-Quantenwells und an GaAs-AlGaAs-GaMnAs-Schichtstrukturen


### 3.1 The start: a fresh state and two messages

Exactly as `Agent.annotate` starts: `new_state` (nothing read, nothing mapped) and the conversation with the
system prompt and the first message.

In [18]:
import time
from openai import BadRequestError
from ie_agent.agent import FORCE_FINISH, NOTHING_MAPPED, Progress
from ie_agent.tools import FINISH

title = docs.title[LIVE_DOC_ID]
p = Progress(LIVE_DOC_ID, new_state(title), [
    {"role": "system", "content": live_agent.system_prompt},
    {"role": "user", "content": live_agent.first_message.format(
        title=title, examples=live_agent.tools.examples(LIVE_DOC_ID, "title"))},
], tracing=False)
step = 0        # index of the next request, as in `for step in range(self.max_steps)`
printed = 0     # how many messages of p.messages were printed already
done = False
start = time.time()
show_state(p.state)

   state: stage title | read [] | mapped 0 | shown by gnd_context 0 | gnd_context calls 0


### 3.2 One pass of the loop

`next_step()` is the body of the `for` loop in `Agent._loop`, with prints in between. Read it side by side with
`agent.py`: the order is the same.

In [19]:
def next_step():
    """One request to the model and what the loop does with the answer (one pass of Agent._loop)."""
    global step, printed, done
    if done:
        print("The document is done: run the result cell (3.4).")
        return
    if step == live_agent.max_steps:
        p.abort, done = "max_steps", True
        print("max_steps reached without an accepted finish.")
        return

    # --- before the request: is finish forced? which tools are offered? ---
    final_step = step == live_agent.max_steps - 1
    forced = final_step or (p.asked_again and bool(p.state["mapped"]))
    if final_step and not p.asked_again:
        p.messages.append({"role": "user", "content": live_agent.last_step})
    offered = offered_tools(p.state, forced, live_agent.examples_with_sections, live_agent.gnd_context,
                            live_agent.toc)

    print(f"==================== REQUEST {step + 1} (at most {live_agent.max_steps}) ====================")
    print(f"MODEL GETS {len(p.messages)} messages; new since the last request:")
    for m in p.messages[printed:]:
        show_message(m)
    if printed == len(p.messages):
        print("   (nothing new besides the tool results printed above)")
    print(f"offered tools: {names(offered)} | " + ("finish FORCED (tool_choice)" if forced else "tool_choice auto"))

    # --- the request ---
    try:
        response = live_agent.client.chat.completions.create(
            model=live_agent.model, messages=p.messages, tools=offered,
            tool_choice=FORCE_FINISH if forced else "auto", temperature=0, max_tokens=live_agent.max_tokens,
            extra_body={"chat_template_kwargs": {"enable_thinking": live_agent.reasoning}})
    except BadRequestError as e:
        if "maximum context length" in str(e):
            p.abort, done = "context_full", True
        elif forced:
            p.abort, p.last_answer, done = "forcing_refused", str(e)[:500], True
        else:
            raise
        print("server refused the request:", p.abort)
        return
    p.tokens_in += response.usage.prompt_tokens
    p.tokens_out += response.usage.completion_tokens
    msg = response.choices[0].message
    calls = msg.tool_calls or []
    live_agent._remember_answer(p, step, msg, calls, offered, forced)   # appends the assistant message
    printed = len(p.messages)
    step += 1

    print()
    print(f"MODEL ANSWERS ({response.usage.prompt_tokens} tokens in, {response.usage.completion_tokens} out):")
    reasoning = getattr(msg, "reasoning_content", None) or getattr(msg, "reasoning", None)
    if reasoning:
        print("reasoning:", reasoning[:REASONING_CHARS] + (" …" if len(reasoning) > REASONING_CHARS else ""))
    if msg.content:
        print("text:", msg.content)
    for c in calls:
        print(f"call: {c.function.name}({c.function.arguments})")

    # --- what the loop does with the answer ---
    print()
    print("WHAT HAPPENS:")
    if not calls:
        p.last_answer = (msg.content or "")[:1000]
        if p.asked_again:
            p.abort, done = "no_final_answer", True
            print("second answer without a tool call in a row: abort (no_final_answer)")
        else:
            p.asked_again = True
            p.messages.append({"role": "user",
                               "content": live_agent.last_step if p.state["mapped"] else NOTHING_MAPPED})
            print("no tool call: the model is asked again (the message comes at the next request)"
                  + (", and finish will be forced" if p.state["mapped"] else ""))
        return
    if not forced:
        p.asked_again = False

    for c in calls:
        try:
            args = json.loads(c.function.arguments or "{}")
        except json.JSONDecodeError:
            args = {}
        print(f"--> {c.function.name}")
        if c.function.name != FINISH:
            live_agent._run_tool(p, c, args)        # check_call, then the tool
            if p.steps[-1].get("refused"):
                print("    REFUSED by check_call: the model gets an explanation instead of a result")
        elif live_agent._finish(p, c, args, forced):
            done = True
            print("    finish accepted:", [gnd.label(i) for i in p.final or []])
            if p.forced and p.final is None:
                print("    (forced, but without a valid ID: the fallback in _result applies)")
            break
        else:
            print("    finish SENT BACK: some IDs did not come from map_to_gnd or gnd_context")
        for m in p.messages[printed:]:              # the tool result: what the model gets next
            show_message(m)
        printed = len(p.messages)
    show_state(p.state)

### 3.3 Step through

Run the next cells one by one; each is one request. Before running the next one, guess what the model will
be offered. If the model needs more requests than there are cells, run the last one again, or the loop cell
for the rest.

In [20]:
next_step()

==================== REQUEST 1 (at most 14) ====================
MODEL GETS 2 messages; new since the last request:
----- SYSTEM -----
Du bist Fachreferent:in für die verbale Sacherschließung an der Deutschen Nationalbibliothek und vergibst Schlagwörter aus der Gemeinsamen Normdatei (GND) für wissenschaftliche Arbeiten.

Du bekommst den Titel eines Dokuments und Beispiele ähnlicher, bereits erschlossener Titel. Gehe stufenweise vor, wie es Erschließer:innen tun:

1. Bestimme aus dem Titel das Hauptthema.
2. Reicht der Titel nicht aus (zu allgemein, metaphorisch, mehrdeutig, Fachbegriffe unklar), lies mit read_abstract_intro() Gegenstand und Fragestellung.
3. Bilde deine Kandidaten aus Titel und Abstract mit map_to_gnd() ab. Fehlt danach noch etwas, lies mit read_toc() Aufbau und behandelte Aspekte. Das Inhaltsverzeichnis steht erst zur Verfügung, wenn du nach dem Abstract abgebildet hast. Lies nur so weit, wie du es brauchst.
4. Die Beispiele zum Titel hast du schon. Mit get_examples()

In [21]:
next_step()

==================== REQUEST 2 (at most 14) ====================
MODEL GETS 4 messages; new since the last request:
   (nothing new besides the tool results printed above)
offered tools: ['get_examples', 'map_to_gnd'] | tool_choice auto

MODEL ANSWERS (3379 tokens in, 758 out):
reasoning: Let me analyze the title and abstract.

Title: "Optische Spektroskopie an zweidimensionalen Lochsystemen in GaAs-AlGaAs-Quantenwells und an GaAs-AlGaAs-GaMnAs-Schichtstrukturen"

Main topic: Optical spectroscopy on two-dimensional hole systems in GaAs-AlGaAs quantum wells and on GaAs-AlGaAs-GaMnAs layer structures.

From the abstract:
- Spintronics context
- Diluted magnetic semiconductor GaMnAs
- Optical spin injection
- Spin relaxation of electrons and holes (a focus of the work)
- Hole spin relaxation in GaAs quantum wells
- D'yakonov-Perel' mechanism
- Optical detection via magneto-optical effects (Kerr effect, Faraday rotation)

So the document is about:
1. Optical spectroscopy (optische Spektros

In [22]:
next_step()

==================== REQUEST 3 (at most 14) ====================
MODEL GETS 6 messages; new since the last request:
   (nothing new besides the tool results printed above)
offered tools: ['read_toc', 'get_examples', 'map_to_gnd', 'gnd_context'] | tool_choice auto

MODEL ANSWERS (4487 tokens in, 1344 out):
reasoning: Let me analyze the results:

1. "Optische Spektroskopie" - 040436802, score 1.0 ✓
2. "Quantenwell" - 041240103, score 1.0 ✓
3. "Galliumarsenid" - 040191559, score 1.0 ✓
4. "Aluminiumgalliumarsenid" - 1135930449, score 1.0 ✓
5. "Galliummanganarsenid" - mapped to Galliumarsenid 040191559, score 0.7 - this is wrong, GaMnAs is a different material. Let me check if there's a specific GND for GaMnAs.
6. "Spinrelaxation" - 041823338, score 1.0 ✓
7. "Spintronik" - 1014244218, score 1.0 ✓
8. "Zweidimensionales System" - mapped to "Zweidimensionales Material" 1216131074, score 0.785 - this might not be the right term. The document is about 2D hole systems in quantum wells, not 2D mat

In [23]:
next_step()

==================== REQUEST 4 (at most 14) ====================
MODEL GETS 9 messages; new since the last request:
   (nothing new besides the tool results printed above)
offered tools: ['read_toc', 'get_examples', 'map_to_gnd', 'finish', 'gnd_context'] | tool_choice auto

MODEL ANSWERS (6462 tokens in, 3375 out):
reasoning: Let me analyze the results.

The title: "Optische Spektroskopie an zweidimensionalen Lochsystemen in GaAs-AlGaAs-Quantenwells und an GaAs-AlGaAs-GaMnAs-Schichtstrukturen"

Main topics:
1. Optische Spektroskopie (optical spectroscopy) - 040436802, score 1.0 ✓
2. Quantenwell (quantum well) - 041240103, score 1.0 ✓
3. GaAs (Galliumarsenid) - 040191559, score 1.0 ✓
4. AlGaAs (Aluminiumgalliumarsenid) - 1135930449, score 1.0, but assigned 0 times in training documents. Hmm.
5. GaMnAs - no good match. "Gallium-Mangan-Arsenid" mapped to Gallium-Indium-Arsenid (wrong, score 1.0 but wrong concept!). "GaMnAs" mapped to GaN (wrong). "Manganarsenid" mapped to Manganarsenide (

In [24]:
next_step()

==================== REQUEST 5 (at most 14) ====================
MODEL GETS 11 messages; new since the last request:
   (nothing new besides the tool results printed above)
offered tools: ['read_toc', 'get_examples', 'map_to_gnd', 'finish', 'gnd_context'] | tool_choice auto

MODEL ANSWERS (7769 tokens in, 2461 out):
reasoning: Let me analyze the document:

Title: "Optische Spektroskopie an zweidimensionalen Lochsystemen in GaAs-AlGaAs-Quantenwells und an GaAs-AlGaAs-GaMnAs-Schichtstrukturen"

Main topics:
1. Optical spectroscopy (Optische Spektroskopie) - the method
2. Two-dimensional hole systems (zweidimensionale Lochsysteme) - the object
3. GaAs-AlGaAs quantum wells (Quantenwells) - the structure
4. GaAs-AlGaAs-GaMnAs layer structures (Schichtstrukturen) - the structure
5. Spin relaxation (Spinrelaxation) - the focus of the work (from abstract: "Die Spinrelaxation von Elektronen und Löchern stellt einen Schwerpunkt dieser Arbeit dar")
6. Spintronics (Spintronik) - the broader contex

In [25]:
next_step()

The document is done: run the result cell (3.4).


In [26]:
while not done:   # the rest at once
    next_step()

### 3.4 The result

`_result` builds the line that `ie-agent run` writes to `log.jsonl`. Without an accepted `finish` (not done yet,
`max_steps`, `context_full`) it falls back to every mapped candidate.

In [27]:
if not done:
    print("not done yet: the result below is the fallback (every mapped candidate)")
if p.forced and p.final is None:
    p.abort = "forced_without_valid_ids"   # as in Agent.annotate
r = live_agent._result(p, time.time() - start)
print({k: r[k] for k in ["read", "n_steps", "n_mapped", "n_context", "rejected", "forced", "abort", "tokens_in",
                         "tokens_out", "seconds"]})
print("assigned:", [s["term"] for s in r["suggestions"]])
score(r["gnd_ids"], LIVE_DOC_ID)

{'read': ['abstract_intro'], 'n_steps': 6, 'n_mapped': 16, 'n_context': 2, 'rejected': 0, 'forced': False, 'abort': None, 'tokens_in': 23606, 'tokens_out': 8260, 'seconds': 180.47}
assigned: ['Optische Spektroskopie', 'Quantenwell', 'Galliumarsenid', 'Spinrelaxation', 'Spintronik', 'Magnetooptischer Effekt']
P 0.33 | R 0.67 | F1 0.44
hits:   ['Galliumarsenid', 'Optische Spektroskopie']
extra:  ['Magnetooptischer Effekt', 'Quantenwell', 'Spinrelaxation', 'Spintronik']
missed: ['Heterostruktur']


## Where to find this in the code

| step | code |
|---|---|
| loading data, mapping | `src/ie_agent/data.py` |
| GND graph, what `gnd_context` shows | `src/ie_agent/gnd/` (`graph.py`, `context.py`) |
| tool definitions the model sees (`TOOLS`, `GND_CONTEXT_TOOL`) | `src/ie_agent/tools.py` |
| stages, `offered_tools`, `check_call` | `src/ie_agent/tools.py`, section *IE policy* |
| `read_*`, `map_to_gnd`, `gnd_context` | `src/ie_agent/tools.py`, `ToolRunner` |
| the loop, forced `finish`, ID check | `src/ie_agent/agent.py`, `Agent.annotate`, `Agent._loop` (Part 3: `next_step`) |
| prompts | `prompts/*.md` |